# Voicebox + Qwen3-TTS for EPUB Player (free Colab trial)
This notebook runs the **actual open-source [jamiepine/voicebox](https://github.com/jamiepine/voicebox)** backend on a Google Colab GPU and gives the iPhone EPUB reader a temporary HTTPS connection.

**Before Run all:** Runtime → Change runtime type → choose a GPU (T4 is fine when Colab offers one). Free GPU availability and session length are controlled by Google and are not guaranteed.

Keep this runtime alive while you listen. When Colab disconnects, run the notebook again and tap the new pairing button. Your generated audiobook chunks remain cached in EPUB Player.

In [ ]:
import os, subprocess, sys, time, pathlib
try:
    import torch
    print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')
except Exception as e:
    print('GPU check:', e)
if not __import__('torch').cuda.is_available():
    raise RuntimeError('No GPU is attached. In Colab choose Runtime → Change runtime type → GPU, then Run all again.')


In [ ]:
# Install the Voicebox backend exactly from the open-source project. First run can take several minutes.
import os, subprocess, sys, pathlib
root = pathlib.Path('/content/voicebox')
if not root.exists():
    subprocess.run(['git','clone','--depth','1','https://github.com/jamiepine/voicebox.git',str(root)], check=True)
# Pin the Voicebox revision used when this EPUB integration was built.
subprocess.run(['git','-C',str(root),'fetch','--depth','1','origin','51f49dea198384b4eb6087b72c17057c6eb1c1cd'], check=True)
subprocess.run(['git','-C',str(root),'checkout','--detach','51f49dea198384b4eb6087b72c17057c6eb1c1cd'], check=True)
sentinel = pathlib.Path('/content/.voicebox_epub_ready')
if not sentinel.exists():
    subprocess.run([sys.executable,'-m','pip','install','-q','-r',str(root/'backend/requirements.txt')], check=True)
    subprocess.run([sys.executable,'-m','pip','install','-q','--no-deps','chatterbox-tts','hume-tada'], check=True)
    subprocess.run([sys.executable,'-m','pip','install','-q','git+https://github.com/QwenLM/Qwen3-TTS.git'], check=True)
    sentinel.write_text('ready')
print('Voicebox backend installed.')


In [ ]:
# Start Voicebox, put a small token-protected proxy in front of it, then make a free HTTPS Quick Tunnel.
import os, subprocess, sys, time, pathlib, secrets, re, urllib.request, textwrap, html
from IPython.display import display, HTML

for name in ('VOICEBOX_PROCESS','VOICEBOX_PROXY','VOICEBOX_TUNNEL'):
    old = globals().get(name)
    if old is not None:
        try: old.terminate()
        except Exception: pass

token = secrets.token_urlsafe(32)
env = os.environ.copy()
env['VOICEBOX_CORS_ORIGINS'] = 'https://epubplayer-eta.vercel.app'
env['PYTHONUNBUFFERED'] = '1'
voicebox_log = open('/content/voicebox-backend.log','w')
VOICEBOX_PROCESS = subprocess.Popen([sys.executable,'-m','backend.main','--host','127.0.0.1','--port','17493','--data-dir','/content/voicebox-data'], cwd='/content/voicebox', env=env, stdout=voicebox_log, stderr=subprocess.STDOUT)

import requests
for _ in range(120):
    try:
        r = requests.get('http://127.0.0.1:17493/health', timeout=2)
        if r.ok: break
    except Exception: pass
    if VOICEBOX_PROCESS.poll() is not None:
        raise RuntimeError(pathlib.Path('/content/voicebox-backend.log').read_text()[-5000:])
    time.sleep(1)
else:
    raise RuntimeError('Voicebox did not start. See /content/voicebox-backend.log')

proxy_code = r'''
import os
import httpx
from fastapi import FastAPI, Request
from fastapi.responses import Response
from fastapi.middleware.cors import CORSMiddleware
TOKEN = os.environ['VOICEBOX_EPUB_TOKEN']
UPSTREAM = 'http://127.0.0.1:17493'
app = FastAPI()
app.add_middleware(CORSMiddleware, allow_origins=['https://epubplayer-eta.vercel.app'], allow_credentials=False, allow_methods=['*'], allow_headers=['*'])
@app.api_route('/{path:path}', methods=['GET','POST','PUT','DELETE','PATCH','OPTIONS'])
async def proxy(path: str, request: Request):
    if request.method != 'OPTIONS' and request.headers.get('x-voicebox-token') != TOKEN:
        return Response('Unauthorized', status_code=401)
    body = await request.body()
    headers = {k:v for k,v in request.headers.items() if k.lower() not in {'host','content-length','x-voicebox-token','origin','referer'}}
    async with httpx.AsyncClient(timeout=600.0) as client:
        resp = await client.request(request.method, f'{UPSTREAM}/{path}', params=request.query_params, content=body, headers=headers)
    out_headers = {k:v for k,v in resp.headers.items() if k.lower() not in {'content-length','content-encoding','transfer-encoding','connection','access-control-allow-origin','access-control-allow-credentials'}}
    return Response(resp.content, status_code=resp.status_code, headers=out_headers, media_type=resp.headers.get('content-type'))
'''
pathlib.Path('/content/voicebox_secure_proxy.py').write_text(proxy_code)
proxy_env = os.environ.copy(); proxy_env['VOICEBOX_EPUB_TOKEN'] = token
proxy_log = open('/content/voicebox-proxy.log','w')
VOICEBOX_PROXY = subprocess.Popen([sys.executable,'-m','uvicorn','voicebox_secure_proxy:app','--host','127.0.0.1','--port','7860'], cwd='/content', env=proxy_env, stdout=proxy_log, stderr=subprocess.STDOUT)
for _ in range(30):
    try:
        if requests.get('http://127.0.0.1:7860/health', headers={'X-Voicebox-Token':token}, timeout=2).ok: break
    except Exception: pass
    time.sleep(1)
else: raise RuntimeError('Secure proxy did not start')

cloudflared = pathlib.Path('/content/cloudflared')
if not cloudflared.exists():
    urllib.request.urlretrieve('https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64', cloudflared)
    cloudflared.chmod(0o755)
VOICEBOX_TUNNEL = subprocess.Popen([str(cloudflared),'tunnel','--url','http://127.0.0.1:7860','--no-autoupdate'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
tunnel_url = None
deadline = time.time() + 45
while time.time() < deadline:
    line = VOICEBOX_TUNNEL.stdout.readline()
    if not line:
        time.sleep(.2); continue
    m = re.search(r'https://[a-zA-Z0-9.-]+\.trycloudflare\.com', line)
    if m:
        tunnel_url = m.group(0); break
if not tunnel_url: raise RuntimeError('Cloudflare tunnel did not provide a URL')

from urllib.parse import urlencode
pair = 'https://epubplayer-eta.vercel.app/app/settings#' + urlencode({'voiceboxUrl': tunnel_url, 'voiceboxToken': token})
print('VOICEBOX SERVER:', tunnel_url)
print('ACCESS TOKEN:', token)
display(HTML(f'''<div style="font-family:-apple-system;padding:18px;border:1px solid #ddd;border-radius:14px"><h2>Voicebox is ready</h2><p>Keep this Colab runtime running while you listen.</p><a href="{html.escape(pair)}" target="_blank" style="display:inline-block;padding:14px 18px;background:#6d5dfc;color:white;text-decoration:none;border-radius:12px;font-weight:700">Connect EPUB Player to Voicebox</a></div>'''))
